In [162]:
import logging

# 1. Sirf yeh ek block kafi hai sab kuch silent karne ke liye
logging.getLogger('pyEPR').setLevel(logging.ERROR)

# Baaki warnings (jaise Pandas ki internal warnings) ke liye agar aap chahein toh
# sirf ek simple line rakh sakte hain:
import warnings
warnings.filterwarnings("ignore")

In [1]:
%load_ext autoreload
%autoreload 2
%config IPCompleter.greedy = True
import numpy as np
import pyEPR as epr
import pandas as pd
#Helps to find the location of the pyEPR package
#epr.__file__


# --- EPR PROJECT INFO  ---

In [164]:
from pathlib import Path
path_to_project='E:\Krishna_wrk\HFSS_Projects'
#print(f'We will find the example project located in\n{path_to_project}')

In [175]:
pinfo = epr.ProjectInfo(project_path = path_to_project,
                        project_name = 'QubitDesign',
                        design_name='Hello',
                        dielectrics_bulk=['Sapphire'],
                        dielectric_surfaces=['interface'],
                        resistive_surfaces= None,
                        seams=None)
                        

In [270]:
# This will check if the app, project, design, and setup are all linked.
# If it returns True, you are good to go.
print(f"Connection Status: {pinfo.check_connected()}")

Connection Status: True


In [167]:
pinfo.junctions['j1']={'Lj_variable' : 'Lj',
                       'rect':'Rectangle3',
                       'line': 'JJ_Line',
                       'length': epr.parse_units('100um')}
pinfo.validate_junction_info()


In [148]:
#Let's see what all objects in the design are pinfo contains a function to retrieve these for convenience.
pinfo.get_all_object_names()

['UPPER_CAGE',
 'Sapphire',
 'LOWER_CAGE',
 'CAVITY',
 'Rectangle1',
 'Rectangle2',
 'Rectangle3',
 'JJ_Line']

In [149]:
pinfo.get_all_variables_names()


['Lj']

In [151]:
#Let's see what the design type was been used to design the object
pinfo.design.solution_type

'Eigenmode'

In [152]:
pinfo.design.get_setup_names()

('Setup1',)

In [153]:
pinfo.setup.get_mesh_stats()

,Unnamed: 0,Num Tets,Min edge length,Max edge length,RMS edge length,Min tet vol,Max tet vol,Mean tet vol,Std Devn (vol)
0,UPPER_CAGE,2329,1.211560000000000e-02,2.831350000000000e+01,7.013270000000000e+00,8.739470000000000e-10,1.951530000000000e+02,8.611150000000000e+00,2.268080000000000e+01
1,Sapphire,1323,4.170860000000000e-02,2.000000000000000e+00,4.975440000000000e-01,3.621820000000000e-06,8.368109999999999e-02,5.442180000000000e-03,1.176660000000000e-02
2,LOWER_CAGE,2969,1.229340000000000e-02,2.705720000000000e+01,6.156620000000000e+00,9.913980000000001e-09,1.906410000000000e+02,6.754710000000000e+00,2.082960000000000e+01
3,CAVITY,13530,1.034960000000000e-02,8.315000000000000e+00,2.169000000000000e+00,3.965400000000000e-10,4.258300000000000e+00,4.442950000000000e-01,4.305000000000000e-01


In [154]:
pinfo.setup.n_modes

'2'

In [155]:
#Calling this function will run an analysis of the design in the background
#The console will wait for HFSS to complete
#This requires optometrics license for HFSS
#The default one will not be repeated
#pinfo.design.optimetrics.solve_setup(pinfo.design.optimetrics.get_setup_names()[0])
#pinfo.design.optimetrics.solve_setup(pinfo.design.optimetrics.get_setup_names())
#remove [0] if no optometrics

In [ ]:
pinfo.setup.analyze()

# ---  EPR DISTRIBUTED ANALYSIS ---

In [157]:
# This is the core object for interacting with HFSS 
# and running analysis within HFSS
eprd = epr.DistributedAnalysis(pinfo) #epr HFSS analysis

Design "Hello" info:
	# eigenmodes    2
	# variations    1


# --- 1. THE MAIN WORKHORSE ---

In [277]:
%%capture
# This runs the EPR math for all modes and variations. 
# It calculates the participation ratios (p_mj) for your junctions.
eprd.do_EPR_analysis(append_analysis=False)

# --- 2. INFORMATION & METADATA ---

In [366]:
# Shows a dict of solved variations (e.g., {'0': "Lj='10nH'"})
#eprd.get_variations()

In [365]:
# Returns a DataFrame of all variables used in the sweep
#eprd.n_variations

In [364]:
# Retrieves all Ansys variables for the design using distributed analysis
#eprd.get_ansys_variables()

In [363]:
# Tells you which variation is the "default" (usually '0')
#eprd.get_nominal_variation_index()

In [362]:
# Returns the exact string Ansys uses for variation 0
#eprd.get_variation_string('0')

In [361]:
# Retrieves previously analyzed variations to avoid redundant calculations
#eprd.get_previously_analyzed()

# --- 3. FREQUENCY & CONVERGENCE ---

In [ ]:
# Check if your simulation actually converged before trusting the data
#eprd.get_convergence('0')  # Returns DataFrame of Max Delta Freq per pass

In [ ]:
#eprd.get_convergence_vs_pass('0')  # Returns freq of each mode at every pass

In [ ]:
#eprd.hfss_report_f_convergence('0') # Creates a convergence plot INSIDE the Ansys UI

In [ ]:
#eprd.hfss_report_full_convergence()  # More detailed plot of convergence

In [359]:
#eprd.get_ansys_frequencies_all() # Returns multi-index DF of Freqs and Q-factors for everything

In [360]:
#eprd.get_freqs_bare_pd('0')  # Get freqs/Qs for a specific variation as a DataFrame

In [ ]:
#eprd.quick_plot_frequencies() # Instant plot of how freqs change across variations

In [358]:
#for variation in eprd.variations[:4]:  # just for the first 2
    #Fs,Qs = eprd.get_freqs_bare_pd(variation=variation, frame=False)
    #display(pd.DataFrame({'Freq. (GHz)':Fs, 'Quality Factor' : Qs}))

# --- 4. JUNCTION & ENERGY CALCULATIONS ---

<h>Energy participation of dielectric in various modes </h>

In [348]:
import pandas as pd

# 1. Setup & Calculate
eprd.set_variation('0')
eprd.set_mode(0)
total_E = eprd.calc_energy_electric(obj='AllObjects')
sub_E   = eprd.calc_energy_electric(obj='Sapphire')
cav_E   = eprd.calc_energy_electric(obj='CAVITY')

# 2. Create DataFrame
# We name the first column "Mode 0" so it becomes the header
data = {
    "Mode 0": ["Sapphire", "Cavity", "TOTAL"],
    "Energy (J)": [sub_E, cav_E, total_E],
    "Distribution (%)": [
        (sub_E / total_E) * 100,
        (cav_E / total_E) * 100,
        (sub_E + cav_E) / total_E * 100
    ]
}

df_final = pd.DataFrame(data)

# 3. Formatting & Hiding the Index
# This is the trick: We hide the numerical index (0, 1, 2)
# So "Mode 0" becomes the very first thing you see on the top left.
format_mapping = {
    "Energy (J)": "{:.4e}",
    "Distribution (%)": "{:.2f}%"
}

display(
    df_final.style
    .format(format_mapping)
    .hide(axis="index") # <--- This removes the empty/numbered column
)

Mode 0,Energy (J),Distribution (%)
Sapphire,6.9401e-23,87.59%
Cavity,9.8352e-24,12.41%
TOTAL,7.9236e-23,100.00%


In [349]:
import pandas as pd

# 1. Setup: Switch to Mode 1 (Resonator)
eprd.set_variation('0')
eprd.set_mode(1) 

# 2. Efficient Calculation
# Step A: Calculate Substrate (and capture E_total for reuse)
# Returns: participation, (Energy_in_obj, Total_Energy_System)
p_sub, (E_sub, E_total) = eprd.calc_p_electric_volume('Sapphire', variation='0')

# Step B: Calculate Vacuum (reuse E_total to skip re-integration)
p_vac, (E_vac, _) = eprd.calc_p_electric_volume('CAVITY', variation='0', E_total=E_total)

# 3. Build Data Dictionary
data = {
    "Mode 1": ["Sapphire", "Vacuum Cavity", "SUM (Check)"],
    "Energy (J)": [E_sub, E_vac, (E_sub + E_vac)],
    "Participation": [p_sub, p_vac, (p_sub + p_vac)],
    "Distribution (%)": [p_sub * 100, p_vac * 100, (p_sub + p_vac) * 100]
}

# 4. Create DataFrame
df_mode1 = pd.DataFrame(data)

# 5. Apply "Beautiful" Formatting
format_mapping = {
    "Energy (J)": "{:.4e}",
    "Participation": "{:.6f}",
    "Distribution (%)": "{:.2f}%"
}

print("--- Mode 1: Efficient Energy Breakdown ---")
display(
    df_mode1.style
    .format(format_mapping)
    .hide(axis="index") # Hides the number column to make 'Mode 1' the clean header
)

--- Mode 1: Efficient Energy Breakdown ---


Mode 1,Energy (J),Participation,Distribution (%)
Sapphire,1.3463e-21,0.040959,4.10%
Vacuum Cavity,3.1523e-20,0.959041,95.90%
SUM (Check),3.2870e-20,1.000000,100.00%


In [357]:
# --- CONFIGURATION ---
pd.options.display.float_format = '{:.4e}'.format
var_id = '0'
# Define your object names map
obj_map = {'Sapphire': 'Sapphire', 'Vacuum Cavity': 'CAVITY'}

# --- FUNCTION: Get Distribution for a Mode ---
def get_mode_distribution(mode_idx):
    eprd.set_variation(var_id)
    eprd.set_mode(mode_idx)
    
    # Efficient calc: Get first object + Total
    p_sub, (E_sub, E_total) = eprd.calc_p_electric_volume('Sapphire', variation=var_id) # Same calculations but using the participation ratio function which is more efficient and gives you the breakdown of electric energy in each region in one go.
    # Get second object (reuse total)
    p_vac, (E_vac, _) = eprd.calc_p_electric_volume('CAVITY', variation=var_id, E_total=E_total)
    
    return {
        'Total Energy (J)': E_total,
        'Sapphire (J)': E_sub,
        'Vacuum (J)': E_vac,
        'Sapphire (%)': p_sub * 100,
        'Vacuum (%)': p_vac * 100,
        'Check Sum (%)': (p_sub + p_vac) * 100
    }

# --- MAIN EXECUTION ---
# 1. Run Analysis for both modes
m0_data = get_mode_distribution(0)
m1_data = get_mode_distribution(1)

# 2. Build the Combined DataFrame
# We pivot the data so Modes are columns, and Components are rows
data = {
    'Metric': [
        'Total Energy (J)', 
        'Energy in Sapphire (J)', 'Energy in Vacuum (J)', 
        'Part. Sapphire (%)', 'Part. Vacuum (%)', 
        'Sum Check (%)'
    ],
    'Mode 0 (Qubit)': [
        m0_data['Total Energy (J)'],
        m0_data['Sapphire (J)'], m0_data['Vacuum (J)'],
        m0_data['Sapphire (%)'], m0_data['Vacuum (%)'],
        m0_data['Check Sum (%)']
    ],
    'Mode 1 (Resonator)': [
        m1_data['Total Energy (J)'],
        m1_data['Sapphire (J)'], m1_data['Vacuum (J)'],
        m1_data['Sapphire (%)'], m1_data['Vacuum (%)'],
        m1_data['Check Sum (%)']
    ]
}

df_combined = pd.DataFrame(data)

# 3. Beautify
# We apply specific formatting to J (Energy) vs % (Participation)
def format_values(val):
    if isinstance(val, float):
        if val > 1e-15 or val == 0: # It's likely percent or large number
             if val > 1000 or val < 0.001: return f"{val:.4e}" # Scientific for Energy
             return f"{val:.2f}%" # Percent for Participation
    return val

# Apply styling
display(
    df_combined.style
    .format({
        'Mode 0 (Qubit)': format_values,
        'Mode 1 (Resonator)': format_values
    })
    .hide(axis="index") # Hide numbers
    .set_caption("⚡DIELECTRIC ENERGY DISTRIBUTION IN EACH MODE ⚡")
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', "#8B00FD"), ('font-size', '14px')]},
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]}
    ])
)

Metric,Mode 0 (Qubit),Mode 1 (Resonator)
Total Energy (J),7.92358558498617e-23,3.28695719595178e-20
Energy in Sapphire (J),6.94006522878173e-23,1.34631244181557e-21
Energy in Vacuum (J),9.83520356204447e-24,3.15232595177023e-20
Part. Sapphire (%),87.59%,4.10%
Part. Vacuum (%),12.41%,95.90%
Sum Check (%),100.00%,100.00%


In [221]:
# Get Josephson junction inductance and capacitance
L_J, C_J = eprd.get_junctions_L_and_C(variation='0')

# Extract scalar values (single junction case)
L_J_val = L_J.iloc[0]
C_J_val = C_J.iloc[0]

print("Josephson junction parameters:")
print(f"  Inductance L_J = {L_J_val:.3e} H ({L_J_val*1e9:.3f} nH)")
print(f"  Capacitance C_J = {C_J_val:.3e} F ({C_J_val*1e15:.3f} fF)")



Josephson junction parameters:
  Inductance L_J = 1.400e-08 H (14.000 nH)
  Capacitance C_J = 2.000e-15 F (2.000 fF)


In [222]:
# Get Josephson junction geometry
length, unit_vec = eprd.get_junc_len_dir('0', 'JJ_Line')

# Print with clear physical meaning and SI units
print("Josephson junction geometry:")
print(f"  Junction length L = {length:.3e} m ({length*1e6:.2f} µm)")
print(f"  Junction direction (unit vector) = {unit_vec}")


Josephson junction geometry:
  Junction length L = 5.000e-05 m (50.00 µm)
  Junction direction (unit vector) = [0.0, 1.0, 0.0]


In [ ]:
# Calculate peak (average) current through the Josephson junction. It is specially designed for JJ's. It is based on surface integral of H field over the junction area.
i_max = eprd.calc_avg_current_J_surf_mag('0', 'Rectangle3', 'JJ_Line')

# Print with clear units and explanation
print(f"Peak junction current |I_max| = {i_max:.3e} A")
print(f"                           = {i_max*1e6:.3f} µA")


Peak junction current |I_max| = 4.573e-09 A
                           = 0.005 µA


In [ ]:
# This is defined in genral for any line, It calculates I using surafce intergal of H.dl. It works on Line integral 
I_complex = eprd.calc_line_current(variation='0', junc_line_name='JJ_Line')

# 3. RESULT: It returns a complex number (Magnitude + Phase)
I_mag = abs(I_complex)

#print(f"Current phasor: {I_complex:.2e} Amps")
# Print with clear units and explanation
print(f"Peak junction current |I_max| = {I_mag:.3e} A")
print(f"                           = {I_mag*1e6:.3f} µA")

Peak junction current |I_max| = 1.660e-09 A
                           = 0.002 µA


In [ ]:
# Calculate junction participation for a single mode (requires U_E and U_H pre-calculated)
# p_j = eprd.calc_p_junction('0', total_H, total_E, [8e-9], [0]) 

In [227]:

# Calculate Q based on specific loss mechanisms
# eprd.get_Qdielectric('MyDielectric', mode=0, variation='0') # Q due to bulk dielectric loss
# eprd.get_Qsurface(mode=0, variation='0', name='MyChipSurface') # Q due to surface "dirt"
# eprd.get_Qseam('MySeamLine', mode=0, variation='0') # Q due to radiation/seam los

# --- 5. LOSS & QUALITY FACTOR (Q) CALCULATIONS (to be continued)---

In [ ]:
mode_idx = 0   # THE MODE YOU ARE STUDYING
variation = '0' # tHE VARIATION YOU ARE STUDYING (e.g., '0' for nominal, '1' for first sweep point, etc.)

In [ ]:
#eprd.calc_Q_external('0', freq_GHz=5.0)  Use this in Driven mode which gives Coupling Q to external ports

In [201]:
# 2. DEFINE PHYSICS PARAMETERS
# Access the internal configuration dictionary
pinfo.options.dielectric_surface_thickness = '3nm'  # Thickness of dirt layer
pinfo.options.dielectric_surface_epsilon   = 10     # Dielectric constant of dirt

# (Optional) You can inspect default loss tangents, but we will often 
# pass these manually to be safe, as shown below.

In [207]:
# Calculate Q based on specific loss mechanisms
eprd.get_Qdielectric('Sapphire', mode=0, variation='0', U_E=None) # Q due to bulk dielectric loss


Calculating Qdielectric_Sapphire for mode 0 (0/1)
p_dielectric_Sapphire_0 = 0.04095923255324668


Qdielectric_Sapphire   2.441451994248202e+07
dtype: float64

In [ ]:
eprd.get_Qsurface(mode=0, variation='0', name='Face124') # Q due to surface "dirt"

In [ ]:
eprd.get_Qseam('JJ_Line', mode=0, variation='0') # Q due to radiation/seam loss


# --- 6. MESH & GEOMETRY ---

In [367]:
#print(eprd.get_mesh_statistics('0'))   # See number of tetrahedrons and mesh quality

# --- 7. UTILITIES & SAVING ---

In [ ]:
#eprd.has_fields('0')                   # Check if HFSS field data is still on disk for variation 0
#eprd.setup_data()                      # Prepares the folder structure for saving
#eprd.save()                            # Saves all analyzed data to a .npz or .csv file
# eprd.load('path_to_data.npz')        # Reloads previous analysis results
#eprd.update_ansys_info()               # Force-refresh data from the Ansys COM interface

# --- EPR QUANTUM ANALYSIS ---

In [ ]:
# Initialize (point it to the file created by DistributedAnalysis)
eprq = epr.QuantumAnalysis(eprd.data_filename)

	 Differences in variations:




In [339]:
%%capture
# 1. THE CORE COMMAND: Analyzes everything
# This calculates f_1, chi, and participation for all variations.
eprq.analyze_all_variations(cos_trunc=8, fock_trunc=7) 

## Why Is There a Factor of 2?

The factor-of-two discrepancy arises **solely from the energy definition used
when integrating the electric fields**.  
Both results are correct — they correspond to **different physical conventions**.

---

### Method 1: `get_ansys_energies`
**(Time-averaged electromagnetic energy)**

This method uses the **standard time-averaged energy** stored in an electric field:

$$
U_{\text{avg}} = \frac{1}{4} \int \epsilon \, |E_{\text{peak}}|^2 \, dV
$$

This is equivalent to the circuit expression:

$$
U_{\text{avg}} = \frac{1}{2} C V_{\text{rms}}^2
$$

This definition is natural for:
- Classical EM simulations (HFSS / Ansys)
- Power and energy flow calculations
- Steady-state sinusoidal fields

---

### Method 2: `analyze_all_variations`
**(Peak energy used in Hamiltonian formulations)**

This method uses the **peak (maximum) stored energy**, common in
Hamiltonian mechanics and circuit quantization:

$$
U_{\text{peak}} = \frac{1}{2} \int \epsilon \, |E_{\text{peak}}|^2 \, dV
$$

This corresponds to:

$$
U_{\text{peak}} = \frac{1}{2} C V_{\text{peak}}^2
$$

This definition is preferred when:
- Writing the Hamiltonian
- Quantizing the circuit
- Matching energies to $\hbar \omega$

---

### Origin of the Factor of 2

For a sinusoidal voltage:

$$
V_{\text{peak}}^2 = 2 V_{\text{rms}}^2
$$

Therefore:

$$
U_{\text{peak}} = 2 U_{\text{avg}}
$$

---

### Key Takeaway

$$
\boxed{
\text{The factor of 2 comes from using }
V_{\text{peak}} \text{ vs } V_{\text{rms}},
\text{ not from an error in field integration.}
}
$$

Both methods are internally consistent — they simply answer
**different physical questions**.


## Energy Terms and Their Physical Meaning

| Column Name      | Symbol                | Meaning | In Plain English |
|------------------|-----------------------|---------|------------------|
| `U_J_inds`       | $U_{LJ}$              | Junction Inductive Energy | Energy stored in the Josephson inductance. This is the *hidden magnetic energy* that HFSS does not capture. |
| `U_J_caps`       | $U_{CJ}$              | Junction Capacitive Energy | Energy stored in the tiny capacitor plates of the Josephson junction itself. |
| `U_H`            | $U_{\text{Magnetic}}$ | Field Magnetic Energy | Energy stored in the vacuum and wires (geometric inductance). Calculated by HFSS. |
| `U_E`            | $U_{\text{Electric}}$ | Field Electric Energy | Energy stored in the large capacitor pads and surrounding vacuum. Calculated by HFSS. |
| `U_tot_ind`      | $U_{\text{Total},L}$  | Total Inductive Energy | Sum of magnetic field energy and junction inductive energy: $U_H + U_{LJ}$. |
| `U_tot_cap`      | $U_{\text{Total},C}$  | Total Capacitive Energy | Sum of electric field energy and junction capacitive energy: $U_E + U_{CJ}$. |
| `U_diff`         | Error                 | Numerical Error | Difference between total inductive and total capacitive energy. Ideally zero at resonance. |


In [335]:
#CLASSICAL ENERGIES: Pulls the energy totals from Ansys
eprq.get_ansys_energies()

U_J_inds                      U_J_caps        U_H        U_E  U_tot_ind  U_tot_cap     U_norm     U_diff
variation mode                                                                                                                                
0         0      {'j1': 3.928891253009124e-23}  {'j1': 9.14826668496987e-25} 2.8896e-25 3.9618e-23 3.9578e-23 4.0533e-23 4.0533e-23 1.1920e-02
          1     {'j1': 1.6582005877281014e-23}   {'j1': 9.6150114089848e-25} 1.6414e-20 1.6435e-20 1.6431e-20 1.6436e-20 1.6436e-20 1.4584e-04

### Total Electric and Magnetic Energy of a Single Mode

For a single electromagnetic mode with angular frequency $\omega$, the
time-averaged electric and magnetic energies are given by

$$
U_E
=
\frac{1}{4}
\int_V
\epsilon(\mathbf{r})
\,\lvert \mathbf{E}_{\text{peak}}(\mathbf{r}) \rvert^2
\, dV
$$

$$
U_H
=
\frac{1}{4}
\int_V
\mu(\mathbf{r})
\,\lvert \mathbf{H}_{\text{peak}}(\mathbf{r}) \rvert^2
\, dV
$$

At resonance for a lossless single mode, the energies are equal:

$$
U_E = U_H
$$


In [344]:
# 1. Setup
modes = [0, 1]
data = []

# 2. Extract Data using your specific methods
for m in modes:
    eprd.set_mode(m)
    
    # Raw calculation as requested
    U_E = eprd.calc_energy_electric(obj='AllObjects')
    U_H = eprd.calc_energy_magnetic(obj='AllObjects')
    
    data.append({
        "Mode": m,
        "Electric Energy (J)": U_E,
        "Magnetic Energy (J)": U_H
    })

# 3. Create DataFrame
df_raw = pd.DataFrame(data).set_index("Mode")

# 4. Format for display
pd.options.display.float_format = '{:.6e}'.format

display(df_raw)

,Electric Energy (J),Magnetic Energy (J)
Mode,,
0,7.923586e-23,5.779243e-25
1,3.286957e-20,3.282874e-20


In [ ]:
# --- CONFIGURATION ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.options.display.float_format = '{:,.4e}'.format  # Scientific notation
var_idx = '0'

# --- 1. EXTRACT RAW DATA ---
# Unpack all 7 matrices
PM, SIGN, Om, EJ, Phi_ZPF, PJ_cap, PM_norm = eprq.get_epr_base_matrices(variation=var_idx)

# Get Classical Ansys Data (Freqs & Q)
ansys_freqs = eprd.get_freqs_bare_pd(variation=var_idx)

# Get Ecs (Charging Energy) - Returns a Series
# We use .values to align with the numpy arrays from get_epr_base_matrices
Ecs = eprq.get_Ecs(variation=var_idx).values

# --- 2. CREATE TABLE 1: "THE MODE PHYSICS REPORT" ---
mode_labels = [f"Mode {m}" for m in range(PM.shape[0])]
junc_labels = [f"Junc {j}" for j in range(PM.shape[1])]

data_modes = {
    'Freq_GHz': ansys_freqs['Freq. (GHz)'].values,
    'Q_Factor': ansys_freqs['Quality Factor'].values,
}

for j, j_name in enumerate(junc_labels):

    
    # Inductive Participation
    data_modes[f'Inductive Part(p_mj)'] = PM[:, j]
    
    # ... inside the loop ...
    
    # Normalized Participation (P_nm)
    data_modes[f'Part Norm (P_nm)'] = PM_norm[:, j]

    # Capacitive Participation
    if j < PJ_cap.shape[1]: 
        data_modes[f'Capacitive_Part'] = PJ_cap[:, j]
    
    # ZPF
    data_modes[f'Phi ZPF (φ0)'] = Phi_ZPF[:, j]
    
    # Sign
    data_modes[f'Sign Matrix'] = SIGN[:, j]

df_modes = pd.DataFrame(data_modes, index=mode_labels)

# --- 3. CREATE TABLE 2: "THE DEVICE PARAMETERS" ---

data_junctions = {
    'Ej_GHz': EJ.flatten(),
    'Ec_GHz': Ecs.flatten(), # Added Ec here
    
    # Calculated Device Properties
    # L_j (nH) ≈ 163.4 / E_J (GHz)
    'Calc_Lj_nH': 163.4 / EJ.flatten(),
    
    # C_j (fF) ≈ 19.37 / E_C (GHz)
    'Calc_Cj_fF': 19.37 / Ecs.flatten()
}

df_junctions = pd.DataFrame(data_junctions, index=junc_labels)

# --- 4. DISPLAY ---
print("\n" + "="*60)
print(" TABLE 1: MODE ANALYSIS (Quantum Parameters)")
print("="*60)
print(" • Freq_GHz: Qubit and Resonator Frequencies")
print(" • Q_Factor: Qubit and Resonator Quality Factors")
print(" • Inductive Part(p_mj): Proportion of magnetic energy of given mode stored in JJ")
print(" • Capacitive_Part: Proportion of electric energy of given mode stored in JJ")
print(" • Phi ZPF (φ0): Magnitude of quantum flux fluctuations in units of Phi_0")
print(" • Sign Matrix: Direction of the quantum field in each junction")
display(df_modes)

print("\n" + "="*60)
print(" TABLE 2: JUNCTION PARAMETERS (Device Physics)")
print("="*60)
print(" • Ej, Ec: Josephson and Charging Energies (GHz)")
print(" • Calc_Lj_nH: Linear Inductance (Derived from Ej)")
print(" • Calc_Cj_fF: Junction Capacitance (Derived from Ec)")
display(df_junctions)


 TABLE 1: MODE ANALYSIS (Quantum Parameters)
 • Freq_GHz: Qubit and Resonator Frequencies
 • Q_Factor: Qubit and Resonator Quality Factors
 • Inductive Part(p_mj): Proportion of magnetic energy of given mode stored in JJ
 • Capacitive_Part: Proportion of electric energy of given mode stored in JJ
 • Phi ZPF (φ0): Magnitude of quantum flux fluctuations in units of Phi_0
 • Sign Matrix: Direction of the quantum field in each junction


,Freq_GHz,Q_Factor,Inductive Part(p_mj),Part Norm (P_nm),Capacitive_Part,Phi ZPF (φ0),Sign Matrix
Mode 0,4.5896e+00,3.3147e+06,9.9279e-01,1.7148e-01,2.2570e-02,4.4173e-01,1
Mode 1,7.2426e+00,4.6525e+03,1.0089e-03,-6.8669e-03,5.8501e-05,-1.7689e-02,-1



 TABLE 2: JUNCTION PARAMETERS (Device Physics)
 • Ej, Ec: Josephson and Charging Energies (GHz)
 • Calc_Lj_nH: Linear Inductance (Derived from Ej)
 • Calc_Cj_pF: Junction Capacitance (Derived from Ec)


,Ej_GHz,Ec_GHz,Calc_Lj_nH,Calc_Cj_fF
Junc 0,1.1676e+01,9.6851e+00,1.3995e+01,2.0000e+00


In [ ]:
#THE MASTER REPORT: Prints a beautiful Markdown table in Jupyter
eprq.report_results(numeric=True)

#### Mode frequencies (MHz)

###### Numerical diagonalization

variation,0
0,4.3548e+03+0.0000e+00j
1,7.2423e+03+0.0000e+00j


#### Kerr Non-linear coefficient table (MHz)

###### Numerical diagonalization

0                      1
variation                                                
0         0 2.4777e+02+0.0000e+00j 6.2738e-01+0.0000e+00j
          1 6.2738e-01+0.0000e+00j 4.3879e-04+0.0000e+00j

In [ ]:
# --- 1. EXTRACT RAW DATA ---
# Unpack all 7 matrices
#PM, SIGN, Om, EJ, Phi_ZPF, PJ_cap, PM_norm = eprq.get_epr_base_matrices(variation=var_idx)

In [294]:
# 3. FREQUENCIES: Get the dressed (quantum) frequencies
#eprq.get_frequencies(numeric=True) # numeric=True gives Numerical Diagonalization results

In [295]:
# 4. CHI MATRIX: Get Anharmonicities (diag) and Cross-Kerr (off-diag)
#eprq.get_chis(numeric=True) 

In [296]:
# To get chi between specific modes m and n:
#eprq.get_chis(m=0, n=0)

In [320]:
# Extracts the Junction's Hamiltonian Parameters:
# Ej = Josephson Energy (Inductive Scale)
# Ec = Charging Energy (Capacitive Scale)
#variation_id = '0'  # Change this to analyze a different variation
#Ecs = eprq.get_Ecs(variation=variation_id)
#Ejs = eprq.get_Ejs(variation=variation_id)


In [282]:
# 9. QUALITY FACTORS: Get the Q for each mode
#eprq.get_quality_factors()

In [ ]:
'''
# 10. FIND VARIATIONS: Filter results by variable values
# Find which variation number corresponds to a specific Lj value
vars_list = eprq.get_variations_of_variable_value(swpvar='Lj', value='10nH')

# 11. MULTI-FILTER: Search variations by multiple parameters
target_vars = eprq.get_variation_of_multiple_variables_value({'Lj': '10nH', 'Cj': '2fF'})

# 12. VARIABLE MAPPING: Convert variation indices ('0', '1') to physical values
val = eprq.get_variable_value(swpvar='Lj', lv=['0', '1'])
vs_var = eprq.get_variable_vs(swpvar='Lj')

# 13. ATTRIBUTE MAPPING: Get any internal attribute versus a swept variable
chis_vs_Lj = eprq.get_vs_variable(swp_var='Lj', attr='chis')

# 16. HAMILTONIAN PLOTS: Shows how frequencies/chis change over a sweep
eprq.plot_hamiltonian_results(swp_variable='Lj')
'''

In [264]:
# 15. VARIATION REPORT: Detailed text summary of a single run
#eprq.full_variation_report(variation='0')
#eprq.full_report_variations(var_list=['0', '1']) # Multiple variations

In [247]:
# 19. GENERIC PLOTTING:
# eprq.plot_results(result_dict, 'Y Axis Label', 'Variable Name', 'X Axis Label')

In [249]:
# 17. QUICK PLOTS: Targeted visuals for specific components
#eprq.quick_plot_chi_alpha(mode1=0, mode2=0)     # Plots anharmonicity of mode 0

In [250]:
#eprq.quick_plot_frequencies(mode=0)             # Freq of mode 0 vs sweep

In [251]:
#eprq.quick_plot_participation(mode=0, junction=0) # EPR of Junc 0 in Mode 0

In [252]:
#eprq.quick_plot_mode(mode=0, junction=0)        # A "Dashboard" for one mode

In [253]:
#18. CONVERGENCE PLOTS: Check if the Ansys mesh was good enough
#eprq.quick_plot_convergence() 

In [265]:
# 20. MESH & CONVERGENCE DATA:
#eprq.get_mesh_tot()                            # Total number of tetrahedra

In [254]:
#eprq.get_convergences_max_delta_freq_vs_pass()  # Delta Freq table

In [255]:
#eprq.get_convergences_tets_vs_pass()           # Mesh growth table

In [266]:
#eprq.get_convergences_max_tets()          # Max mesh size reached

In [258]:
# 21. MISC UTILITIES:
#eprq.print_info()                # Prints general status of the QA object


In [267]:
#eprq.print_variation('0')        # Prints raw data for one variation

In [268]:
#eprq.print_result(result)        # Formats a result dictionary into readable text
# eprq.plotting_dic_x(Var_dic, 'Lj') # Internal helper for organizing plot data

In [ ]:
# 1. Get the Chi Matrix (ND)
df_chis = eprq.get_chis(numeric=True)

# 2. Check if variation 0 exists as a string or an integer
# This prevents the KeyError: 0
available_vars = df_chis.index.get_level_values('variation').unique()
var_to_use = 0 if 0 in available_vars else '0'

# 3. Extract the variation and convert to real components
# This removes the "variation" label and the extra zero column from your screenshot
df_chis_var = df_chis.xs(var_to_use, level='variation').apply(lambda x: np.real(x))

# 4. Rename rows and columns to "Mode 0, Mode 1, Mode 2..."
mode_labels = [f"Mode {i}" for i in range(len(df_chis_var))]
df_chis_var.index = mode_labels
df_chis_var.columns = mode_labels

# 5. Display with clean formatting
pd.options.display.float_format = '{:.4f}'.format 

print(f"--- Chi Matrix  (MHz) ---")
display(df_chis_var)

--- Chi Matrix  (MHz) ---


,Mode 0,Mode 1
Mode 0,247.7726,0.6274
Mode 1,0.6274,0.0004


In [ ]:
#pinfo.disconnect()